In [ ]:
import os
import cv2
import numpy as np
import random

# =========================
# CONFIG
# =========================
INPUT_DIR = "original"
OUTPUT_DIR = "augmented_dataset"
TARGET_PER_CLASS = 1000

In [ ]:
# =========================
# AUGMENTATION FUNCTIONS
# =========================

def random_rotation(image):
    angle = random.uniform(-2, 2)  # small realistic rotation
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), borderValue=(255,255,255))
    return rotated

def random_ink_density(image):
    # Simulate ink variation by adjusting contrast/brightness
    alpha = random.uniform(0.6, 1.4)  # contrast
    beta = random.randint(-10, 10)    # brightness
    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    return adjusted

def slight_thickness_variation(image):
    kernel_size = random.choice([1,2])
    kernel = np.ones((kernel_size,kernel_size), np.uint8)

    if random.random() > 0.5:
        image = cv2.dilate(image, kernel, iterations=1)
    else:
        image = cv2.erode(image, kernel, iterations=1)
    return image

In [ ]:
# =========================
# MAIN AUGMENT LOOP
# =========================

os.makedirs(OUTPUT_DIR, exist_ok=True)

for class_name in os.listdir(INPUT_DIR):
    class_path = os.path.join(INPUT_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    print(f"\nProcessing class: {class_name}")

    output_class_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(output_class_path, exist_ok=True)

    images = [f for f in os.listdir(class_path)
              if f.endswith('.png')]

    if len(images) == 0:
        print(f"⚠ No images found in {class_name}. Skipping...")
        continue

    count = 0
    original_imgs = []

    for img_name in images:
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            print(f"⚠ Skipping corrupted image: {img_name}")
            continue

        save_path = os.path.join(output_class_path, f"orig_{count}.png")
        cv2.imwrite(save_path, img)

        original_imgs.append(img)
        count += 1

    if len(original_imgs) == 0:
        print(f"⚠ All images failed to load in {class_name}. Skipping...")
        continue

    remaining = TARGET_PER_CLASS - count
    print(f"Generating {remaining} augmented images...")

    while count < TARGET_PER_CLASS:
        img = random.choice(original_imgs).copy()

        img = random_rotation(img)
        img = random_ink_density(img)
        img = slight_thickness_variation(img)

        save_path = os.path.join(output_class_path, f"aug_{count}.png")
        cv2.imwrite(save_path, img)

        count += 1

print("\n✅ Augmentation Complete!")

In [ ]:
import os
from PIL import Image

dir = "./original"
for folder in os.listdir(dir):
    for file in os.listdir(os.path.join(dir, folder)):
        if file.lower().endswith(".jpg"):
            jpg_path = os.path.join(dir, folder, file)
            png_path = os.path.join(dir, folder, os.path.splitext(file)[0] + ".png")

            try:
                with Image.open(jpg_path) as img:
                    img.save(png_path, "PNG")
                os.remove(jpg_path)
                print(f"Converted and removed: {file}")
            except Exception as e:
                print(f"Failed {file}: {e}")


In [2]:
import os
import glob
import cv2
import numpy as np

from pre_processing import img_processing

INPUT_DIR = "./augmented_dataset"
OUTPUT_DIR = "./dataset_processed"
IMG_SIZE = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_images = 0

print("Starting offline preprocessing...\n")

for class_name in os.listdir(INPUT_DIR):

    class_input_path = os.path.join(INPUT_DIR, class_name)

    if not os.path.isdir(class_input_path):
        continue

    class_output_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    image_paths = glob.glob(os.path.join(class_input_path, "*.*"))

    print(f"Processing class: {class_name} ({len(image_paths)} images)")

    for img_path in image_paths:

        img = cv2.imread(img_path)

        if img is None:
            print(f"Skipping unreadable image: {img_path}")
            continue

        processed = img_processing(img)

        processed = cv2.resize(
            processed,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )

        processed = cv2.cvtColor(processed, cv2.COLOR_GRAY2BGR)

        filename = os.path.basename(img_path)
        output_path = os.path.join(class_output_path, filename)

        cv2.imwrite(output_path, processed)

        total_images += 1

print("\n✅ Done!")
print("Total processed images:", total_images)

Starting offline preprocessing...

Processing class: ස (1000 images)
Processing class: ප (1000 images)
Processing class: ර (1000 images)
Processing class: න (1000 images)
Processing class: ත (1000 images)
Processing class: හ (1000 images)
Processing class: ල (1000 images)
Processing class: ක (1000 images)
Processing class: ම (1000 images)
Processing class: ග (1000 images)

✅ Done!
Total processed images: 10000
